### Week 9: Databases and Python

**Today you will:** create a SQLite database from Python, add data safely, ask it questions in SQL, and chart the answer.

Start by picking up this week's material (`Week1/updateClonedRepo.ipynb`). New to everyone, so the whole class works through the same notebook. `sqlite3` comes with Python, so there is nothing to install.

In [ ]:
# Run this cell first. It lets the notebook find the course data folder,
# whether it runs from the week folder or the course folder.
import sys; sys.path.append(".."); from coursetools import data_file

### Concept 1: Create a table and add rows

A **database** stores data in **tables**: rows and named columns, like a spreadsheet that programs can query. SQLite keeps the whole database in one file.

| Step | Code |
| --- | --- |
| connect, which creates the file if needed | `sqlite3.connect("lab.db")` |
| run SQL | `cursor.execute(...)` |
| save the changes | `connection.commit()` |

Put values in with `?` placeholders, never with an f-string. With an f-string, a value containing SQL can change what your query does. This is called **SQL injection**, and placeholders prevent it.

In [ ]:
import sqlite3

connection = sqlite3.connect("lab.db")
cursor = connection.cursor()

cursor.execute("CREATE TABLE IF NOT EXISTS samples (name TEXT, mass REAL, volume REAL)")
cursor.execute("DELETE FROM samples")      # start empty each time you run this cell

samples = [("Copper", 89.6, 10.0), ("Granite", 54.0, 20.0), ("Ice", 9.2, 10.0)]
for name, mass, volume in samples:
    cursor.execute("INSERT INTO samples VALUES (?, ?, ?)", (name, mass, volume))

connection.commit()
print("Saved")

**Now break it.** Remove `IF NOT EXISTS` and run the cell twice. The second time you get an `OperationalError`: the table is already there. Put it back.

### Concept 2: Asking questions with SELECT

`SELECT` asks the database a question. `WHERE` filters rows, and `ORDER BY` sorts them. SQL can do the maths too. `fetchall()` gives the answer as a list of tuples.

In [ ]:
cursor.execute("SELECT name, mass / volume FROM samples WHERE mass / volume > 1 ORDER BY mass / volume DESC")

for name, density in cursor.fetchall():
    print(f"{name}: {density:.2f} g/cm3")

Ice is missing: its density is below 1, which is why it floats.

**Now break it.** Change `DESC` to `ASC`. Then take out the whole `WHERE` part. You change the question without touching the Python.

### Concept 3: The weather data in a database

Load the whole CSV into a table once. After that, questions that would each need their own loop become one line of SQL. `GROUP BY` collects rows together, and `AVG` averages each group.

In [ ]:
import csv
import matplotlib.pyplot as plt

cursor.execute("CREATE TABLE IF NOT EXISTS weather (station TEXT, year INTEGER, month INTEGER, tmax REAL, rain REAL)")
cursor.execute("DELETE FROM weather")

with open(data_file("uk_weather.csv")) as file:
    reader = csv.reader(file)
    next(reader)
    for row in reader:
        if row[3] != "" and row[6] != "":
            cursor.execute("INSERT INTO weather VALUES (?, ?, ?, ?, ?)",
                           (row[0], int(row[1]), int(row[2]), float(row[3]), float(row[6])))
connection.commit()

cursor.execute("SELECT month, AVG(rain) FROM weather WHERE station = ? GROUP BY month", ("Lerwick",))
months = []
rain = []
for month, average in cursor.fetchall():
    months.append(month)
    rain.append(average)

plt.bar(months, rain)
plt.xlabel("Month")
plt.ylabel("Average rainfall (mm)")
plt.title("Lerwick, every year since 1930")
plt.show()

Nearly a century of Shetland weather, averaged in one query. Rows with a gap are skipped when loading, which is the Week 5 lesson again.

`("Lerwick",)` has a comma because the values must be a tuple, even when there is only one.

**Now break it.** Change `"Lerwick"` to `"Lerwik"`. No error, just an empty chart: SQL found no rows. Always check a query actually returned something.

### Now the challenges

Save this notebook with Ctrl and S, then open **`week9Challenges.ipynb`**.